# Создание и заполнение данных БД Postgre

In [ ]:
%pip install python-dotenv sqlalchemy psycopg2-binary pandas ipython-sql

In [ ]:
from dotenv import load_dotenv
import pandas as pd
# загрузка переменных
load_dotenv()

### Подключение к PostgreSQL

In [ ]:
from sqlalchemy.engine import URL
import os

host = os.getenv("DB_HOST") or 'localhost'  # если переменная не установлена, используем localhost
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")

conn = URL.create(
    "postgresql",
    username=user,
    password=password,
    host=host,
    database=db,
    port=port
).render_as_string(hide_password=False)


### Техническая функция чтобы удобно было вызывать sql запросы (вариант без sql ячеек)


In [ ]:
def execute_sql(string: str, connection_string = conn):
    df = pd.read_sql_query(string, con=connection_string)
    return df

#### Вот как может использоваться


In [ ]:
execute_sql(
    "SELECT * "
    "FROM user_logs "
    "LIMIT 10",
)

### Далее буду использовать sql-ячейки потому-что так удобнее

In [ ]:
%%sql
SELECT *
FROM user_logs
LIMIT 10;

Вам предоставлена БД с логами (действиями) студентов на образовательном портале за весенний семестр (агрегация по каждой неделе) по отдельному электронному курсу - таблица user_logs (примечание. создана в предыдущих л.р.).
- сourseid — уникальный идентификатор курса, дисциплины;
- userid — уникальный идентификатор студента (не используется в обучении);
- num_week — номер недели в году;
- s_all — количество всех событий на текущий момент;
- s_all_avg — среднее количество всех событий в неделю;
- s_course_viewed — количество просмотров курса;
- s_course_viewed_avg — среднее количество просмотров курса в неделю;
- s_q_attempt_viewed — количество просмотров теста;
- s_q_attempt_viewed_avg — среднее количество просмотров теста в неделю;
- s_a_course_module_viewed — количество просмотров модуля в курсе;
- s_a_course_module_viewed_avg — среднее количество просмотров модуля в курсе в неделю;
- s_a_submission_status_viewed — количество отправленных заданий на проверку;
- s_a_submission_status_viewed_avg — среднее количество ответов;
- namer_level — оценка за дисциплину;
- depart — номер кафедры;
- name_osno — основа обучения (имеет два значения: бюджет или контракт);
- name_formopril — форма обучения;
- leveled — уровень образования (имеет два значения: бакалавриат, магистратура);
- num_sem — номер семестра;
- kurs — номер курса учебной группы.

Также в таблице  departments хранятся названия кафедр, таблица связана с логами по полю depart:
id - код кафедры;
name - сокращенное название кафедры. 

## Задание 1 (если до этого еще этот шаг не был выполнен):

Измените данные вещественного типа, сейчас целая и дробная часть разделены запятой, замените ее на точку. 

Выведите первые 10 записей, чтобы проверить результат предобработки.

### Шаг был выполнен в лабораторной работе 7


## Задание 2: 

Выведите количество кафедр, за которыми закреплены курсы на портале.





In [ ]:
%%sql
SELECT -- количество уникальных кафедр
    COUNT(DISTINCT depart)
FROM user_logs;

##  Задание 3:

Выведите сколько у каждой кафедры закреплено электронных курсов на портале. 
Требуется выводит сокращенное название кафедры и количество курсов. 
У какой кафедры больше всего курсов на портале?

In [ ]:
%%sql
SELECT departments.name, COUNT(DISTINCT courseid) as course_sum
FROM user_logs
JOIN departments ON departments.id = user_logs.depart
GROUP BY departments.name
ORDER BY course_sum DESC

Как мы можем видеть кафедра ДиСО имеет больше всего курсов

## Задание 4:

Ответьте на вопрос: существуют ли курсы, за которыми закреплено несколько кафедр? Если такие курсы есть, то выведите их количество.
Также выведите названия кафедр, которые совместно преподают один и тот же курс.




In [ ]:
%%sql
SELECT COUNT(*) AS multicursi
FROM (
    SELECT courseid
    FROM public.user_logs
    GROUP BY courseid
    HAVING COUNT(DISTINCT depart) > 1
) AS subquery;

Получаем кафедры которые совместно ведут курсы

In [ ]:
%%sql
WITH multicurs as
    (SELECT
    courseid,
    COUNT(DISTINCT dep.name) AS dep_count,
    string_agg(DISTINCT dep.name, ', ')
FROM
    user_logs
JOIN
    departments AS dep ON depart = dep.id
GROUP BY
    courseid
HAVING
    COUNT(DISTINCT dep.name) > 1
ORDER BY
    dep_count DESC
LIMIT 1
)
SELECT AVG(user_logs.namer_level)
FROM user_logs
WHERE courseid = (
SELECT courseid FROM multicurs
                )


## Задание 5:

Выведите количество студентов, которые получили 2, 3, 4, 5.

Пример вывода:

| namer_level |	count |
|-----|------|
|2 |	4 |
|3 |	3435 |
|4 | 	4676765|
|5 | 232 |


In [ ]:
%%sql
SELECT
    COUNT(DISTINCT userid),
    namer_level as оценка
FROM user_logs
GROUP BY namer_level

In [ ]:
%%sql
SELECT COUNT(DISTINCT user_logs.courseid)
FROM user_logs

## Задание 6:

Выведите студента, который больше всех работает на портале (у него максимальное количество логов за вест период обучения).

*на сколько я понял имеется ввиду максимальное число записей с 1 userid*

*нет видимо это просто s_all*


In [ ]:
%%sql
WITH s_all_max_stud as (
SELECT
    userid,
    SUM(s_all) AS s_all_max
FROM user_logs
GROUP BY userid
ORDER BY s_all_max desc
LIMIT 1
)
SELECT avg(user_logs.namer_level)
FROM user_logs
WHERE userid = (select userid from s_all_max_stud)

## Задание 7:

Выведите по каждой недели среднее количество всех событий на портале.

In [ ]:
%%sql
SELECT
    user_logs.num_week,
    AVG(user_logs.s_all)
FROM user_logs
GROUP BY num_week;

## Задание 8: 

Выведите название кафедры, у которой больше всего отличников.

Отдельно выведите название кафедры, у которой больше всего двоечников.

In [ ]:
%%sql
SELECT category, name, student_count
FROM (
    SELECT
        CASE WHEN u.namer_level = 5 THEN 'Отличники' ELSE 'Двоечники' END AS category,
        d.name,
        COUNT(DISTINCT u.userid) AS student_count,
        ROW_NUMBER() OVER (
            PARTITION BY (CASE WHEN u.namer_level = 5 THEN 'Отличники' ELSE 'Двоечники' END)
            ORDER BY COUNT(DISTINCT u.userid) DESC
        ) as rank
    FROM user_logs u
    JOIN departments d ON d.id = u.depart
    WHERE u.namer_level IN (2, 5)
    GROUP BY category, d.name
) AS ranked_stats
WHERE rank = 1;

In [ ]:
%%sql
(
SELECT
    'Отличники' AS category,
    departments.name,
    COUNT(DISTINCT userid) AS student_count
    FROM user_logs
    JOIN departments ON departments.id = user_logs.depart
    WHERE namer_level = 5
    GROUP BY departments.name
    ORDER BY student_count DESC
    LIMIT 1
)

UNION ALL

(
SELECT
    'Двоечники' AS category,
    departments.name,
    COUNT(DISTINCT userid) AS student_count
    FROM user_logs
    JOIN departments ON departments.id = user_logs.depart
    WHERE namer_level = 2
    GROUP BY departments.name
    ORDER BY student_count DESC
    LIMIT 1
);

In [ ]:
%%sql
SELECT count(DISTINCT userid) AS dvoechnkiki, departments.name
FROM user_logs
JOIN departments ON departments.id = user_logs.depart
WHERE namer_level = 2
GROUP BY departments.name
ORDER BY dvoechnkiki DESC
LIMIT 1

## Задание 9:
Провести анализ пиковой активности студентов перед экзаменом (с использованием (Common Table Expression — CTE), оператор with).

Вывести, на какой неделе семестра студенты проявляли наибольшую активность в курсе в целом, и как эта активность распределяется между студентами-бюджетниками и контрактниками.

Пример вывода :

| name_osno | week_number	| avg_s_all	| avg_s_course_viewed |	avg_s_q_attempt_viewed |
|-----|------|------|------|------|
| бюджет |	14	| 125.45 |	45.67 |	32.12 |
|контракт |	14	| 98.76 |	38.90 |	25.43 |

In [ ]:
%%sql
WITH week_activity AS (
    -- Средняя активность по неделям
    SELECT
        CASE
            WHEN name_osno = 1 THEN 'бюджет'
            WHEN name_osno = 2 THEN 'контракт'
            ELSE 'не указано'
        END AS name_osno,
        num_week,
        AVG(s_all) AS avg_as_all,
        AVG(s_course_viewed) AS avg_s_course_viewed,
        AVG(s_q_attempt_viewed) AS s_q_attempt_viewed_avg
    FROM user_logs
    GROUP BY name_osno, num_week
),
week AS (
    -- Находим неделю, где s_all_avg был самым высоки
    SELECT num_week
    FROM user_logs
    GROUP BY num_week
    ORDER BY AVG(s_all) DESC
    LIMIT 1
)
-- данные пиковой недели
SELECT * FROM week_activity
WHERE num_week = (SELECT num_week FROM week)
ORDER BY name_osno;